# Learned light verifier + ILRC on MEVID-SC (no LLM, CPU)

Step 2 of the plan. A **small neural network on pairs of fingerprints** (inputs: |a-b|, a*b and cosine similarity of the two CSCI or tracker fingerprints; one hidden layer of 128) replaces the hand-made checks. It's trained **only on the 104 MEVID-SC train people**. It's still lightweight: under a minute to train, milliseconds per decision.

- **Cell 3:** trains the verifier per backbone (CSCI, the OSNet baseline). It scores both test groups and, by a 2-fold cross-fit, the train people out-of-sample, so they can serve as an extra calibration pool.
- **Cell 4, fair comparison:** at the same delivered precision, does the learned verifier give fewer IDs per person than CSCI's similarity?
- **Cell 5, ILRC v2** with three calibration pools (the other test group of 27 people, the 104 train people, and both), against the point-threshold baseline, tested on the held-out group.

**Needs** the train fingerprints from `MEVID-SC_Train_Fingerprints.ipynb`. A normal CPU runtime is enough; about 5 to 10 minutes. Results go to `MEVID-results/LearnedVerifier-SameClothes`.

In [ ]:
# 1. Load train + test fingerprints for both backbones, and the groups
import os, glob, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0] + '  (run the train-fingerprints '
                            'notebook first)')

CS = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
           'train_templates_csci_sc.npz',
           f'{SD}/**/CSCI-SameClothes/train_templates_csci_sc.npz'])
TR = find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
           'train_templates_tracker_sc.npz',
           f'{SD}/**/Tracker-SameClothes/train_templates_tracker_sc.npz'])
OUT = os.path.join(os.path.dirname(CS), 'LearnedVerifier-SameClothes')
os.makedirs(OUT, exist_ok=True)


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


te = np.load(f'{CS}/test_templates_csci_sc.npz')
tr = np.load(f'{CS}/train_templates_csci_sc.npz')
tte = np.load(f'{TR}/test_templates_tracker_sc.npz')
ttr = np.load(f'{TR}/train_templates_tracker_sc.npz')
pid, cam = te['pid'], te['cam']                  # test clips
pid_tr, cam_tr = tr['pid'], tr['cam']            # train clips
assert (tte['pid'] == pid).all() and (ttr['pid'] == pid_tr).all()
assert not set(pid) & set(pid_tr), 'train and test people overlap!'
DATA = {'CSCI': (unit(tr['templates']), unit(te['templates'])),
        'OSNet baseline (OSNet)': (unit(ttr['zs']), unit(tte['zs']))}
proto = json.load(open(f'{CS}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])

SETTINGS = {'hidden': [128], 'hard_negatives': 10,
            'random_negatives': 10, 'max_positives': 30000,
            'person_error_limit': 0.05, 'delta': 0.10,
            'alphas': [0.05, 0.10], 'bootstrap': 2000,
            'pair_precision_target': 0.95, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print(f'Train: {len(pid_tr)} clips, {len(np.unique(pid_tr))} people | '
      f'Test: {len(pid)} clips, {len(np.unique(pid))} people | '
      'no overlap')
print('Results go to', OUT)


In [ ]:
# 2. The learned light verifier: a small network on pairs of
#    fingerprints, trained ONLY on train people (never on test people)
from sklearn.neural_network import MLPClassifier
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment


def pair_feats(A, B):
    """|a-b| and a*b (full fingerprint) plus cosine similarity: a
    learned, identity-agnostic weighting of the fingerprint dimensions."""
    return np.hstack([np.abs(A - B), A * B,
                      (A * B).sum(1, keepdims=True)]).astype(np.float32)


def train_verifier(X, p, seed):
    """positives: same-person pairs; negatives: each clip's hardest
    different-person clips plus random ones."""
    rng = np.random.default_rng(seed)
    Z = X
    S = Z @ Z.T
    pos = [(i, j) for q in np.unique(p)
           for a, i in enumerate(np.flatnonzero(p == q))
           for j in np.flatnonzero(p == q)[a + 1:]]
    if len(pos) > SETTINGS['max_positives']:
        pos = [pos[k] for k in rng.choice(len(pos),
                                          SETTINGS['max_positives'],
                                          replace=False)]
    neg = []
    for i in range(len(p)):
        other = np.flatnonzero(p != p[i])
        hard = other[np.argsort(-S[i, other])[:SETTINGS['hard_negatives']]]
        rand = rng.choice(other, SETTINGS['random_negatives'], replace=False)
        neg += [(i, j) for j in np.concatenate([hard, rand])]
    pairs = np.array(pos + neg)
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    Xp = pair_feats(Z[pairs[:, 0]], Z[pairs[:, 1]])
    m = MLPClassifier(hidden_layer_sizes=tuple(SETTINGS['hidden']),
                      batch_size=2048, early_stopping=True,
                      n_iter_no_change=3, max_iter=25, random_state=seed)
    m.fit(Xp, y)
    return m


def prob_matrix(m, X, chunk=20000):
    Z = X
    n = len(Z)
    iu = np.triu_indices(n, 1)
    P = np.zeros((n, n), np.float32)
    vals = np.empty(len(iu[0]), np.float32)
    for s in range(0, len(vals), chunk):
        a, b = iu[0][s:s + chunk], iu[1][s:s + chunk]
        vals[s:s + chunk] = m.predict_proba(pair_feats(Z[a], Z[b]))[:, 1]
    P[iu] = vals
    P = P + P.T
    np.fill_diagonal(P, 1)
    return P


def _cont(pred, p):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1))
    np.add.at(M, (pi, ci), 1)
    return M, pi, ci


def person_losses(pred, p):
    """per person: share of its same-ID pairs that are other people
    (computed from the person x ID count table)."""
    M, _, _ = _cont(pred, p)
    nc = M.sum(0)
    tot = (M * (nc - 1)).sum(1)
    wrong = (M * (nc - M)).sum(1)
    return np.where(tot > 0, wrong / np.maximum(tot, 1), 0.0)


def score(pred, p, c):
    M, pi, ci = _cont(pred, p)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0)
    tp = (M * (M - 1)).sum()
    same_id = (nc * (nc - 1)).sum()
    # same person, different camera: ordered pairs, total and same ID
    _, ki = np.unique(c, return_inverse=True)
    K = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(K, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (K.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (K ** 2).sum((1, 2))).sum()
    lp = person_losses(pred, p)
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': (100 * tp / same_id if same_id
                                 else np.nan),
            'cross-camera recall %': 100 * x_same / max(1, x_tot),
            'mean wrong-link share %': 100 * lp.mean(),
            'people failing %':
                100 * (lp > SETTINGS['person_error_limit']).mean()}


GRID = np.round(np.arange(0.01, 0.951, 0.01), 2)


def sweep(D, p, c, with_scores=True):
    """global IDs at every merge distance: losses (+ scores)."""
    D = np.clip((D + D.T) / 2, 0, None)       # rounding can give -1e-7
    Z = linkage(D[np.triu_indices(len(D), 1)], 'average')
    losses, scores = {}, {}
    for d in GRID:
        lab = fcluster(Z, d, 'distance')
        losses[d] = person_losses(lab, p)
        if with_scores:
            scores[d] = score(lab, p, c)
    return losses, scores


In [ ]:
# 3. Train the verifiers and build global IDs at every threshold.
#    Each backbone's result is SAVED TO DRIVE as soon as it is done, so a
#    restart never loses finished work and cells 4-5 can load it.
import pickle, gc, psutil
rng = np.random.default_rng(SETTINGS['seed'])
tr_people = np.unique(pid_tr)
half = set(rng.choice(tr_people, len(tr_people) // 2, replace=False))
h = np.array([q in half for q in pid_tr])
print('Fingerprint size:', {k: v[0].shape[1] for k, v in DATA.items()},
      '| free RAM: %.1f GB' % (psutil.virtual_memory().available / 1e9))
RUN = {}
for bb, (Xtr, Xte) in DATA.items():
    ck = f"{OUT}/run_{bb.split()[0]}.pkl"
    if os.path.exists(ck):
        RUN[bb] = pickle.load(open(ck, 'rb'))
        print(f'{bb}: loaded finished result from Drive')
        continue
    t0 = time.time()
    Xtr, Xte = Xtr.astype(np.float32), Xte.astype(np.float32)
    full = train_verifier(Xtr, pid_tr, SETTINGS['seed'])
    print(f'  {bb}: verifier trained ({time.time() - t0:.0f} s)')
    R = {'test': {}, 'train': {}}
    for g in 'AB':
        ig = np.flatnonzero(fold == g)
        S = Xte[ig] @ Xte[ig].T
        Pm = prob_matrix(full, Xte[ig])
        R['test'][g] = {
            'similarity': sweep(1 - S, pid[ig], cam[ig]),
            'learned verifier': sweep(1 - Pm, pid[ig], cam[ig])}
        del S, Pm; gc.collect()
        print(f'  {bb}: test group {g} done ({time.time() - t0:.0f} s)')
    pool = {'similarity': {d: [] for d in GRID},
            'learned verifier': {d: [] for d in GRID}}
    for part in [h, ~h]:
        m = train_verifier(Xtr[~part], pid_tr[~part], SETTINGS['seed'] + 1)
        ip = np.flatnonzero(part)
        S = Xtr[ip] @ Xtr[ip].T
        for sname, D in [('similarity', 1 - S),
                         ('learned verifier',
                          1 - prob_matrix(m, Xtr[ip]))]:
            losses, _ = sweep(D, pid_tr[ip], cam_tr[ip], with_scores=False)
            for d in GRID:
                pool[sname][d].append(losses[d])
            del D; gc.collect()
        del S, m; gc.collect()
    R['train'] = {s_: {d: np.concatenate(v[d]) for d in GRID}
                  for s_, v in pool.items()}
    RUN[bb] = R
    pickle.dump(R, open(ck, 'wb'))
    print(f'{bb}: done in {time.time() - t0:.0f} s and saved to Drive')

rows = []
for bb, R in RUN.items():
    for g, byscore in R['test'].items():
        for sname, (_, scores) in byscore.items():
            for d, s_ in scores.items():
                rows.append({'backbone': bb, 'group': g, 'score': sname,
                             'distance': d, **s_})
SW = pd.DataFrame(rows)
SW.to_csv(f'{OUT}/sweep_test_groups.csv', index=False)
print('Saved the sweep to', OUT)


In [ ]:
# 4. FAIR COMPARISON at matched precision (diagnostic): does the learned
#    verifier give fewer IDs per person than CSCI's similarity?
import matplotlib.pyplot as plt
rows = []
for (bb, g, sname), d in SW.groupby(['backbone', 'group', 'score'],
                                    sort=False):
    for lvl in [90, 95]:
        ok = d[d['pair precision %'] >= lvl]
        if len(ok):
            b = ok.loc[ok['IDF1 %'].idxmax()]
            rows.append({'backbone': bb, 'precision level %': lvl,
                         'score': sname, 'group': g, 'IDF1 %': b['IDF1 %'],
                         'IDs per person': b['IDs per person'],
                         'cross-camera recall %':
                             b['cross-camera recall %']})
fair = (pd.DataFrame(rows).groupby(['backbone', 'precision level %',
                                    'score'], sort=False)
        [['IDF1 %', 'IDs per person', 'cross-camera recall %']]
        .mean().round(1))
fair.to_csv(f'{OUT}/fair_comparison.csv')
print('AT MATCHED PRECISION (mean of groups A and B)')
display(fair)

fig, axes = plt.subplots(1, len(RUN), figsize=(12, 4.5))
for ax, bb in zip(np.atleast_1d(axes), RUN):
    for (sname, g), d in SW[SW.backbone == bb].groupby(['score', 'group']):
        d = d.dropna(subset=['pair precision %'])
        ax.plot(d['pair precision %'], d['IDs per person'],
                '-' if sname == 'learned verifier' else '--',
                label=f'{sname} (group {g})')
    ax.set_title(bb); ax.set_xlabel('delivered pair precision %')
    ax.set_ylabel('IDs per person (lower is better)')
    ax.set_xlim(70, 100.5); ax.set_ylim(0.9, 12); ax.grid(alpha=.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT}/curve_learned_vs_similarity.png', dpi=150)
plt.show()


In [ ]:
# 5. ILRC v2 with three calibration pools, tested on the held-out group:
#    'group'        : the other test group (27 people, same cameras)
#    'train'        : the 104 train people (other cameras, out-of-sample)
#    'group + train': both (131 people)
#    Rule: most permissive threshold whose person-bootstrap 90% upper
#    bound on the mean wrong-link share stays <= alpha (fixed sequence,
#    strict -> permissive, stop at the first failure). Baseline: the point
#    threshold (pair precision >= 95% on the calibration group).
B, rows = SETTINGS['bootstrap'], []
rng = np.random.default_rng(SETTINGS['seed'])


def ilrc(loss_by_d, alpha):
    n = len(next(iter(loss_by_d.values())))
    idx = rng.integers(0, n, size=(B, n))
    best = bound = None
    for d in GRID:
        ucb = np.quantile(loss_by_d[d][idx].mean(1), 1 - SETTINGS['delta'])
        if ucb > alpha:
            break
        best, bound = d, ucb
    return best, bound


def evaluate(scores, d, ev):
    if d is None:
        ie = np.flatnonzero(fold == ev)
        return score(np.arange(len(ie)), pid[ie], cam[ie])
    return scores[d]


for bb, R in RUN.items():
    for cal, ev in [('A', 'B'), ('B', 'A')]:
        for sname in ['similarity', 'learned verifier']:
            cal_losses, cal_scores = R['test'][cal][sname]
            _, ev_scores = R['test'][ev][sname]
            # baseline: point threshold on the calibration group
            d = None
            for x in GRID:
                pp = cal_scores[x]['pair precision %']
                if pp == pp and pp >= 100 * SETTINGS['pair_precision_target']:
                    d = x
            e = evaluate(ev_scores, d, ev)
            rows.append({'backbone': bb, 'score': sname,
                         'rule': 'point threshold', 'pool': 'group',
                         'alpha %': None, 'direction': f'{cal}->{ev}',
                         'promised %': None, **e})
            pools = {'group': cal_losses,
                     'train': R['train'][sname],
                     'group + train': {x: np.concatenate(
                         [cal_losses[x], R['train'][sname][x]])
                         for x in GRID}}
            for pname, L in pools.items():
                for a in SETTINGS['alphas']:
                    d, bnd = ilrc(L, a)
                    e = evaluate(ev_scores, d, ev)
                    rows.append({'backbone': bb, 'score': sname,
                                 'rule': 'ILRC', 'pool': pname,
                                 'alpha %': int(100 * a),
                                 'direction': f'{cal}->{ev}',
                                 'promised %': None if bnd is None
                                 else round(100 * bnd, 1), **e})
res = pd.DataFrame(rows)
res['held'] = np.where(res['alpha %'].isna(), np.nan,
                       (res['mean wrong-link share %']
                        <= res['alpha %'].fillna(0)).astype(float))
res.to_csv(f'{OUT}/ilrc_learned_verifier_results.csv', index=False)
cols = ['IDF1 %', 'IDs per person', 'pair precision %',
        'mean wrong-link share %', 'cross-camera recall %', 'held']
summ = (res.groupby(['backbone', 'score', 'rule', 'pool', 'alpha %'],
                    dropna=False, sort=False)[cols].mean().round(2))
summ.to_csv(f'{OUT}/ilrc_learned_verifier_summary.csv')
print('HELD-OUT GROUP, mean of A->B and B->A ("held" = share of the two '
      'directions where the promise held)')
display(summ)
